In [ ]:
import torch.optim as optim
from torch.utils.data import DataLoader
import torch.nn as nn
import copy
import torch

def training_helper(model_class,
                   train_data, val_data,
                   lr, batch_size,
                   epochs=100, patience=5,
                   arch_params=None):
  if arch_params:
    model = model_class(**arch_params)
  else:
    model = model_class()

  model.apply(initialize_weights)
  model = model.to(device)

  train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
  val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False)

  loss_fn = nn.CrossEntropyLoss()
  optimizer=optim.SGD(model.parameters(), lr=lr)

  #training loop with early stopping
  train_loss_hist, val_loss_hist = [], []
  train_acc_hist, val_acc_hist = [], []
  best_val_loss = float('inf')
  patience_counter = 0
  improvement_threshold = 1e-3
  best_model_weights = None

  for epoch in range(epochs):
    #Training
        model.train()
        running_loss, correct_train, total_train = 0, 0, 0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = loss_fn(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total_train += labels.size(0)
            correct_train += (predicted == labels).sum().item()

        avg_train_loss = running_loss / len(train_loader)
        avg_train_acc = 100 * correct_train / total_train
        train_loss_hist.append(avg_train_loss)
        train_acc_hist.append(avg_train_acc)

        #Validation
        model.eval()
        running_val_loss, correct_val, total_val = 0, 0, 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = loss_fn(outputs, labels)

                running_val_loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)
                total_val += labels.size(0)
                correct_val += (predicted == labels).sum().item()

        avg_val_loss = running_val_loss / len(val_loader)
        avg_val_acc = 100 * correct_val / total_val
        val_loss_hist.append(avg_val_loss)
        val_acc_hist.append(avg_val_acc)

        # Early Stopping
        improvement = best_val_loss - avg_val_loss
        if improvement > improvement_threshold:
            best_val_loss = avg_val_loss
            patience_counter = 0
            # Save the best model state
            best_model_weights = copy.deepcopy(model.state_dict())
        else:
            patience_counter += 1

        if patience_counter >= patience:
            # print(f"Early stopping at epoch {epoch + 1}") # Optional print
            break

    # Load the best weights before returning
  if best_model_weights:
        model.load_state_dict(best_model_weights)

    # Return the training history and the best model (for C2 comparison)
  return train_loss_hist, val_loss_hist, train_acc_hist, val_acc_hist, model

NameError: name 'nn' is not defined

In [ ]:
#Learning Rate Analysis
import matplotlib.pyplot as plt
import pandas as pd

lrs = [0.001, 0.01, 0.1, 1.0]
lr_results = {}
baseline_batch_size = 64
baseline_epochs = 50

print("Starting Learing rate analysis")
for lr in lrs:
  train_loss, val_loss, train_acc, val_acc, _ = training_helper(
        FeedForwardNN,
        train_dataset, val_dataset,
        lr=lr,
        batch_size=baseline_batch_size,
        epochs=baseline_epochs
    )
  lr_results[lr] = {
        'train_loss': train_loss,
        'val_loss': val_loss,
        'train_acc': train_acc,
        'val_acc': val_acc,
        'epochs_ran': len(val_loss)
    }
  print("Learning rate analysis Finished")


In [ ]:
#Get the best learning rate
best_val_acc = 0.0
best_lr = 0.0

for lr, results in lr_results.items():
    current_best_acc = max(results['val_acc'])

    if current_best_acc > best_val_acc:
        best_val_acc = current_best_acc
        best_lr = lr

In [ ]:
#Linear Curve Plot for each learning rate
import matplotlib.pyplot as plt

plt.figure(figsize=(14, 6))

# Validation Loss Plot
plt.subplot(1, 2, 1)
for lr, results in lr_results.items():
    epochs_ran = results['epochs_ran']
    plt.plot(range(1, epochs_ran + 1), results['val_loss'], label=f'LR={lr}', alpha=0.8)
plt.title('Validation Loss vs. Epochs (Learning Rate Analysis)')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# Validation Accuracy Plot
plt.subplot(1, 2, 2)
for lr, results in lr_results.items():
    epochs_ran = results['epochs_ran']
    plt.plot(range(1, epochs_ran + 1), results['val_acc'], label=f'LR={lr}', alpha=0.8)
plt.title('Validation Accuracy vs. Epochs (Learning Rate Analysis)')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

Analysis: Convergence Speed and Stability


In [ ]:
#Batch Size Analysis
import matplotlib.pyplot as plt
import pandas as pd

batch_sizes = [16, 32, 64, 128]
bs_results={}
baseline_epochs=50

print("Starting Batch Size Analysis")

for bs in batch_sizes:
    train_loss, val_loss, train_acc, val_acc, _ = training_helper(
        FeedForwardNN, 
        train_dataset, val_dataset, 
        lr=best_lr,         
        batch_size=bs,             
        epochs=baseline_epochs
    )
    bs_results[bs] = {
        'val_loss': val_loss, 
        'val_acc': val_acc,
        'epochs_ran': len(val_loss)
    }
print("Batch Size Analysis finished")

In [ ]:
#Get best batch size
best_val_acc_bs = 0.0
best_bs = 0

for bs, results in bs_results.items():
    current_best_acc = max(results['val_acc'])

    # Update best_bs if the current run performed better
    if current_best_acc > best_val_acc_bs:
        best_val_acc_bs = current_best_acc
        best_bs = bs

In [ ]:
#Flexible Neural Network Model
import torch.nn as nn
import torch

class FlexibleFeedForwardNN(nn.Module):
    def __init__(self, num_hidden_layers, neurons_per_layer):
        super().__init__()
        self.flatten = nn.Flatten()
        
        layers = []
        input_size = 28 * 28 
        #Dynamically create hidden layers
        for i in range(num_hidden_layers):
            output_size = neurons_per_layer
            layers.append(nn.Linear(input_size, output_size))
            layers.append(nn.ReLU()) 
            input_size = output_size 
            
        layers.append(nn.Linear(input_size, 10))
        
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        x = self.flatten(x)
        return self.network(x)

In [ ]:
import pandas as pd

layer_tests = [2, 3, 4, 5]     
neuron_tests = [64, 128, 256, 512] 

arch_results = {}
baseline_epochs = 50
print("Starting Architecture Analysis")

fixed_neurons = 128
for layers in layer_tests:
    
    # Define parameters for the custom architecture
    arch_params = {'num_hidden_layers': layers, 'neurons_per_layer': fixed_neurons}
    
    # Run experiment using the FlexibleFeedForwardNN class
    train_loss, val_loss, train_acc, val_acc, _ = training_helper(
        FlexibleFeedForwardNN, # <-- Use the flexible model
        train_dataset, val_dataset, 
        lr=best_lr, 
        batch_size=best_bs, 
        epochs=baseline_epochs,
        arch_params=arch_params 
    )
    
    arch_key = f'{layers}-Layer, {fixed_neurons}-Neuron'
    arch_results[arch_key] = {
        'Layers': layers,
        'Neurons': fixed_neurons,
        'Final Val Acc (%)': max(val_acc),
        'Final Val Loss': min(val_loss),
        'Epochs Run': len(val_loss)
    }

fixed_layers = 3
for neurons in neuron_tests:
    
    # Define parameters for the custom architecture
    arch_params = {'num_hidden_layers': fixed_layers, 'neurons_per_layer': neurons}
    
    # Run experiment
    train_loss, val_loss, train_acc, val_acc, _ = training_helper(
        FlexibleFeedForwardNN, 
        train_dataset, val_dataset, 
        lr=best_lr, 
        batch_size=best_bs, 
        epochs=baseline_epochs,
        arch_params=arch_params
    )
    
    arch_key = f'{fixed_layers}-Layer, {neurons}-Neuron'
    arch_results[arch_key] = {
        'Layers': fixed_layers,
        'Neurons': neurons,
        'Final Val Acc (%)': max(val_acc),
        'Final Val Loss': min(val_loss),
        'Epochs Run': len(val_loss)
    }

print("Architecture Analysis Done")

In [ ]:
#Architecture comparison table
import pandas as pd

# Convert results dictionary to a DataFrame for easy formatting
arch_table_data = []
for key, data in arch_results.items():
    arch_table_data.append(data)

df_arch = pd.DataFrame(arch_table_data)

# Sort by accuracy to identify the best architecture
df_arch = df_arch.sort_values(by='Final Val Acc (%)', ascending=False)

# --- Create Architecture Comparison Table ---
print("\n### Architecture Comparison Table (C1.3)")
# The to_markdown() function prints a nicely formatted table
print(df_arch.to_markdown(index=False, floatfmt=".4f"))

# Extracting the Best Architecture parameters for Part C2
best_arch_row = df_arch.iloc[0]
best_layers_final = int(best_arch_row['Layers'])
best_neurons_final = int(best_arch_row['Neurons'])
best_arch_val_acc = best_arch_row['Final Val Acc (%)']